In [1]:
import os
import warnings
import logging

# ===== SILENCE ALL WARNINGS =====
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

# Suppress Python warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Suppress TensorFlow logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

# Now import
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/lstm-method/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

# TensorFlow imports with suppressed logging
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib
from scipy.stats import uniform, loguniform, randint

# Final suppression of retracing warnings (these are annoying but harmless)
tf.keras.backend.clear_session()

E0000 00:00:1775936620.655317      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775936620.717779      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775936621.219954      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775936621.220000      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775936621.220003      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775936621.220005      16 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_used = 'gru'

model_run_name = f'{model_used}_v2'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report   = False
save_model  = False
random_search = True
train_size = 0.2
pipe_name = 'temporal_extractor'

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=train_size,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)
num_pattern  = 'acc|rot|thm|tof'

# In your notebook, after defining columns
raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns
)

preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), make_column_selector(pattern="acc|rot|thm|tof")),
], remainder="drop", verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

sequence_builder = utils.SequencePadder(maxlen=60, padding_value=-999.0)  # ← your new window size

rnn_clf = utils.KerasRNNClassifier(
    rnn_type=model_used,
)

classifier = utils.ManyToOneWrapperRNN(estimator=rnn_clf, target="gesture_action")

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ("preprocessor", preprocessor),
    ("sequence_builder", sequence_builder),
    ("classifier", classifier),
])

cv = GroupKFold(n_splits=3)

if random_search:
    param_grid = {
        f'{pipe_name}__acc_mode': ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode': ['delta'],
        f'{pipe_name}__tof_mode': ['baseline'],
        # Sequence length - capture different gesture durations
        'sequence_builder__maxlen': randint(10, 120),
        'sequence_builder__padding_value': [-999.0],
        
        # RNN architecture size
        'classifier__estimator__rnn_type': [model_used],
        'classifier__estimator__rnn_units': [ (256, 128)],
        'classifier__estimator__dense_units': [(16,),],
        
        # Regularization to match model size
        'classifier__estimator__dropout': uniform(0.4, 0.42),
        'classifier__estimator__learning_rate': loguniform(1e-4, 5e-3),
        
        # Keep these fixed
        'classifier__estimator__bidirectional': [True],
        'classifier__estimator__class_weight_mode': ['balanced'],
        'classifier__estimator__epochs': [100],
        'classifier__estimator__patience': [10],
        'classifier__estimator__batch_size': [16],
    }

else:
    param_grid = {
        # RawSequenceExtractor params
        f'{pipe_name}__acc_cols': [acc_columns],
        f'{pipe_name}__rot_cols': [rot_columns],
        f'{pipe_name}__thm_cols': [thm_columns],
        f'{pipe_name}__tof_cols': [tof_columns],
        f'{pipe_name}__acc_mode': ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode': ['delta'],
        f'{pipe_name}__tof_mode': ['baseline'],

        # SequencePadder params
        'sequence_builder__maxlen': [120],
        'sequence_builder__padding_value': [-999.0],

        # RNN params (nested under classifier__estimator__)
        'classifier__estimator__rnn_type': [model_used],
        'classifier__estimator__rnn_units': [(128,), (128, 32)],
        'classifier__estimator__dense_units': [(32, 16)],
        'classifier__estimator__dropout': [0.1, 0.14],
        'classifier__estimator__learning_rate': [1e-4],
        'classifier__estimator__batch_size': [16],
        'classifier__estimator__epochs': [120],
        'classifier__estimator__patience': [10],
        "classifier__estimator__bidirectional": [True],
        "classifier__estimator__class_weight_mode": ["balanced"],
    }

In [6]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=5,  
                cv=cv,
                random_state=42,
                n_jobs=1,  
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=1,              # safer with tensorflow/keras on Kaggle
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']
        
        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
    
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


In [7]:
master_cv_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__estimator__batch_size,param_classifier__estimator__bidirectional,param_classifier__estimator__class_weight_mode,param_classifier__estimator__dense_units,param_classifier__estimator__dropout,param_classifier__estimator__epochs,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score,orientation_data,model_target
0,43.201903,10.084561,1.563151,0.055644,16,True,balanced,"(16,)",0.557307,100,...,0.272480,0.079566,5,0.514851,0.39,0.319588,0.408146,0.080742,Lie on Back,gesture_action
1,49.444220,4.762355,1.737553,0.271509,16,True,balanced,"(16,)",0.727470,100,...,0.294774,0.035756,2,0.475248,0.42,0.422680,0.439309,0.025436,Lie on Back,gesture_action
2,53.191242,18.852372,1.898175,0.486233,16,True,balanced,"(16,)",0.441989,100,...,0.285474,0.130655,3,0.742574,0.40,0.824742,0.655772,0.183943,Lie on Back,gesture_action
3,30.490848,2.776558,1.485017,0.042841,16,True,balanced,"(16,)",0.460004,100,...,0.435276,0.097502,1,0.762376,0.75,0.701031,0.737802,0.026488,Lie on Back,gesture_action
4,23.701490,4.376406,1.432013,0.023767,16,True,balanced,"(16,)",0.807362,100,...,0.282487,0.037301,4,0.247525,0.31,0.432990,0.330171,0.077048,Lie on Back,gesture_action
